# Building Multimodal RAG Application — Part 8: Putting It All Together

This Colab-ready companion notebook assembles the complete maintained workflow: multimodal record preparation, joint retrieval, prompt processing, open-source LVLM inference, and evidence-aware output through LangChain Core runnables.

> **Archive note:** The original article used Prediction Guard, BridgeTower, and LanceDB. This maintained notebook preserves the architecture while using current open-source components that run without an API key. The compact LVLM is suitable for demonstration; switch to a larger checkpoint for production quality.


## 1. Install dependencies


In [ ]:
!pip install -q "langchain-core>=1.0" "sentence-transformers>=3.0" "transformers>=4.52" accelerate pillow requests


## 2. Imports and configuration


In [ ]:
from io import BytesIO

import requests
import torch
from IPython.display import display
from PIL import Image
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

CLIP_MODEL_ID = "clip-ViT-B-32"
LVLM_MODEL_ID = "llava-hf/llava-interleave-qwen-0.5b-hf"
RUN_LVLM = True


## 3. Preprocess the multimodal records

Each record represents a frame produced by the video-preprocessing stage together with its aligned transcript. The notebook downloads three public images, normalizes them to RGB, and keeps the evidence fields needed by retrieval and generation.


In [ ]:
records = [
    {
        "id": "astronaut",
        "image_url": "https://cdn-images-1.medium.com/max/800/1*Jlado1XvMOgrgGXYVD1wQg.png",
        "transcript": (
            "As I look back on the mission aboard the International Space Station, "
            "I'm proud to have been part of the science activities during the last two months."
        ),
    },
    {
        "id": "skateboard",
        "image_url": "https://farm4.staticflickr.com/3300/3497460990_11dfb95dd1_z.jpg",
        "transcript": "A skateboarder performs an inverted trick in a concrete skate park.",
    },
    {
        "id": "diagram",
        "image_url": (
            "https://huggingface.co/datasets/huggingface/documentation-images/"
            "resolve/main/transformers/tasks/ai2d-demo.jpg"
        ),
        "transcript": "A scientific diagram contains numbered labels and arrows.",
    },
]

def load_image(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

for record in records:
    record["image"] = load_image(record["image_url"])

print(f"Prepared {len(records)} multimodal records.")


## 4. Build the multimodal retrieval module

CLIP embeds the frame images and the user query in a shared space. The retriever returns the best frame plus its transcript and similarity score, representing the vector-store stage of the production architecture.


In [ ]:
clip_model = SentenceTransformer(CLIP_MODEL_ID)
image_embeddings = clip_model.encode(
    [record["image"] for record in records],
    convert_to_tensor=True,
    normalize_embeddings=True,
)

def retrieve_frame(query):
    query_embedding = clip_model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(query_embedding, image_embeddings)[0]
    best_index = int(torch.argmax(scores))
    result = dict(records[best_index])
    result["score"] = float(scores[best_index])
    return result

retriever_module = RunnableLambda(retrieve_frame)


## 5. Build prompt processing and LVLM inference


In [ ]:
def prepare_lvlm_input(inputs):
    retrieved = inputs["retrieved_results"]
    prompt = (
        "Answer the question using the retrieved image and transcript. "
        "Treat the transcript as authoritative evidence.\n"
        f"Transcript: {retrieved['transcript']}\n"
        f"Question: {inputs['user_query']}\n"
        "Respond with one complete, specific sentence."
    )
    return {
        "image": retrieved["image"],
        "image_url": retrieved["image_url"],
        "record_id": retrieved["id"],
        "retrieval_score": retrieved["score"],
        "prompt": prompt,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image", "image": retrieved["image_url"]},
                {"type": "text", "text": prompt},
            ],
        }],
    }

prompt_processing_module = RunnableLambda(prepare_lvlm_input)


In [ ]:
generator = None
if RUN_LVLM:
    generator = pipeline(
        task="image-text-to-text",
        model=LVLM_MODEL_ID,
        device_map="auto",
        dtype=(torch.float16 if torch.cuda.is_available() else torch.float32),
    )

def run_lvlm(payload):
    if generator is None:
        return "[LVLM preview] " + payload["prompt"]
    result = generator(
        text=payload["messages"],
        max_new_tokens=80,
        do_sample=False,
        return_full_text=False,
    )
    generated = result[0]["generated_text"]
    return generated[-1]["content"] if isinstance(generated, list) else generated

lvlm_inference_module = RunnableLambda(run_lvlm)


## 6. Compose the end-to-end Multimodal RAG chain


In [ ]:
mm_rag_chain = (
    RunnableParallel({
        "retrieved_results": retriever_module,
        "user_query": RunnablePassthrough(),
    })
    | prompt_processing_module
    | RunnableParallel({
        "final_text_output": lvlm_inference_module,
        "input_to_lvlm": RunnablePassthrough(),
    })
)


## 7. Invoke the system and inspect its evidence


In [ ]:
query = "What do the astronauts feel about their work?"
response = mm_rag_chain.invoke(query)

print(f"Indexed {len(records)} multimodal records.")
print(f"Retrieved record: {response['input_to_lvlm']['record_id']}")
print(f"Retrieval similarity: {response['input_to_lvlm']['retrieval_score']:.4f}")
print(f"\nUSER Query: {query}")
print(f"MM-RAG Response: {response['final_text_output']}")
display(response["input_to_lvlm"]["image"])


## 8. Production extensions

- Replace the in-memory CLIP matrix with LanceDB or another persistent vector store.
- Feed frames produced by the Part 4 preprocessing pipeline instead of the three public samples.
- Upgrade the compact LVLM and evaluate grounded-answer quality on a held-out question set.
- Preserve retrieved evidence, similarity scores, and model metadata for observability.
- Add access control, batching, caching, and monitoring before serving the chain in production.
